### MEASURE: evaluation objective metrics code

In [ ]:
from __future__ import annotations
import os
import json
import random
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple
import numpy as np
import soundfile as sf
import librosa
import torch
try:
    from jiwer import wer
except ImportError:
    wer = None

_SPEECHBRAIN_AVAILABLE = False
try:
    import torch
    from speechbrain.inference.speaker import EncoderClassifier
    _SPEECHBRAIN_AVAILABLE = True
except Exception:
    _SPEECHBRAIN_AVAILABLE = False
_FASTER_WHISPER_AVAILABLE = False
try:
    from faster_whisper import WhisperModel
    _FASTER_WHISPER_AVAILABLE = True
except Exception:
    _FASTER_WHISPER_AVAILABLE = False

In [ ]:
def load_audio_mono(path: str, target_sr: int = 16000) -> Tuple[np.ndarray, int]:
    y, sr = sf.read(path, always_2d=False)
    if y.ndim == 2:
        y = np.mean(y, axis=1)
    y = y.astype(np.float32)
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        sr = target_sr
    y = np.nan_to_num(y)
    return y, sr


def cosine_similarity(a: np.ndarray, b: np.ndarray, eps: float = 1e-9) -> float:
    a = a.reshape(-1).astype(np.float32)
    b = b.reshape(-1).astype(np.float32)
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) + eps
    return float(np.dot(a, b) / denom)


def embedding_mfcc_mean(y: np.ndarray, sr: int, n_mfcc: int = 20) -> np.ndarray:
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    emb = mfcc.mean(axis=1)
    return emb.astype(np.float32)


class SpeakerEmbedder:
    def __init__(self, device: Optional[str] = None):
        self.device = device
        self.model = None
        if _SPEECHBRAIN_AVAILABLE:
            if device is None:
                device = "cuda" if torch.cuda.is_available() else "cpu"
            self.device = device
            self.model = EncoderClassifier.from_hparams(
                source="speechbrain/spkrec-ecapa-voxceleb",
                run_opts={"device": self.device})

    def embed(self, y: np.ndarray, sr: int) -> np.ndarray:
        if self.model is None:
            return embedding_mfcc_mean(y, sr)
        wav = torch.from_numpy(y).to(self.device).unsqueeze(0)
        with torch.no_grad():
            emb = self.model.encode_batch(wav).squeeze(0).squeeze(0).detach().cpu().numpy()
        return emb.astype(np.float32)

def f0_rmse(y_ref: np.ndarray, y_syn: np.ndarray, sr: int = 16000,
            fmin: float = 50.0, fmax: float = 500.0) -> float:
    f0_ref, _, _ = librosa.pyin(y_ref, fmin=fmin, fmax=fmax, sr=sr)
    f0_syn, _, _ = librosa.pyin(y_syn, fmin=fmin, fmax=fmax, sr=sr)
    L = min(len(f0_ref), len(f0_syn))
    f0_ref = f0_ref[:L]
    f0_syn = f0_syn[:L]
    mask = np.isfinite(f0_ref) & np.isfinite(f0_syn)
    if mask.sum() == 0:
        return float("nan")
    diff = f0_ref[mask] - f0_syn[mask]
    return float(np.sqrt(np.mean(diff ** 2)))

def transcribe_whisper(audio_path: str, model_size: str = "small") -> str:
    if not _FASTER_WHISPER_AVAILABLE:
        raise RuntimeError("faster-whisper is not installed. Install: pip install faster-whisper")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    model = WhisperModel(model_size, device=device, compute_type=compute_type)

    segments, _info = model.transcribe(audio_path, beam_size=5)
    text = " ".join(seg.text.strip() for seg in segments).strip()
    return text

def compute_wer(ref_text: str, hyp_text: str) -> float:
    if wer is None:
        raise RuntimeError("jiwer is not installed. Install: pip install jiwer")
    return float(wer(ref_text, hyp_text))

@dataclass
class AudioMetricsResult:
    cosine_sim: float
    f0_rmse: float
    wer: Optional[float]
    ref_asr: Optional[str]
    syn_asr: Optional[str]

def evaluate_pair(
    ref_wav_path: str,
    syn_wav_path: str,
    ref_text: Optional[str] = None,
    use_asr: bool = True,
    asr_model_size: str = "small",
) -> AudioMetricsResult:
    y_ref, sr = load_audio_mono(ref_wav_path, target_sr=16000)
    y_syn, _ = load_audio_mono(syn_wav_path, target_sr=16000)
    embedder = SpeakerEmbedder()
    emb_ref = embedder.embed(y_ref, sr)
    emb_syn = embedder.embed(y_syn, sr)
    cos = cosine_similarity(emb_ref, emb_syn)
    f0 = f0_rmse(y_ref, y_syn, sr=sr)
    ref_asr = None
    syn_asr = None
    W = None
    if use_asr:
        syn_asr = transcribe_whisper(syn_wav_path, model_size=asr_model_size)
        if ref_text is not None:
            ref_asr = None
            W = compute_wer(ref_text, syn_asr)
        else:
            ref_asr = transcribe_whisper(ref_wav_path, model_size=asr_model_size)
            W = compute_wer(ref_asr, syn_asr)
    return AudioMetricsResult(
        cosine_sim=cos,
        f0_rmse=f0,
        wer=W,
        ref_asr=ref_asr,
        syn_asr=syn_asr)
